# Homework - Logistic Regression implementation with Spark

In this homework you will be implementing logistic regression to classify text documents. The implementation will be in Python, on top of Spark.

Here are the steps you will follow for this classification task.   
1. Feature standardization  
2. Learning logistic regression model through gradient descent
3. Evaluating the learned model

## Data

You will be dealing with a training set that consists of around 170,000 text documents and a test/evaluation
set that consists of around 18,700 text documents. All but around 6,000 of these text documents are Wikipedia
pages; the remaining documents are descriptions of Australian court cases and rulings. At the highest level,
your task is to build a classifier that can automatically figure out whether a text document is an Australian
court case.

We have prepared four datasets for your use. Two of them are the entire training set and test set that are counted as "big" files and are stored in S3. Two of them are subsets of training set and test set that are counted as "small" files and are stored in the `data` directory of your Vocareum workspace.

* The entire training set (~ 1.8 GB of text): `s3://comp643bucket/homework/spark_logreg/TrainingDataOneLinePerDoc.txt`
* The entire test set (~ 200 MB of text): `s3://comp643bucket/homework/spark_logreg/TestingDataOneLinePerDoc.txt`
* The small subset of training set : `data/SmallTrainingDataOneLinePerDoc.txt`
* The small subset of test set : `data/SmallTestingDataOneLinePerDoc.txt`

As before, you should start running and debugging your code on the small datasets through this Jupyter Notebook, and then run your code in AWS EMR cluster on the entire big datasets.  

The training and test sets are text files that every line in the file represents a document. Take a look at the small training set `SmallTrainingDataOneLinePerDoc.txt`. You'll see that each document begins with a `<doc id = ...>` tag, and ends with `</doc>`. The texual body of each document is stored inside the `<doc>` tag.

Note that all of the Australia legal cases begin with something like `<doc id = "AU..." ...>`;
that is, the doc id for an Australian legal case always starts with `AU`. Other documents from Wikipedia have a pure numeric doc id. Therefore, doc id indicates the "label" or "class" of each document; i.e., whether it's an Australian court case or not.

You will be learning your logistic regression model based on this "labeled" training set. Then, the goal is to have a model that can predict if the document is an Australian legal case by **only** looking at the contents of the document.

In [ ]:
!pip install pyspark

In [ ]:
import pyspark
import re
import numpy as np

In [ ]:
# pyspark works best with java8
# set JAVA_HOME enviroment variable to java8 path
%env JAVA_HOME = /usr/lib/jvm/java-8-openjdk-amd64

In [ ]:
sc = pyspark.SparkContext()

## Provided code - creating reference dictionary

This provided code is the same as what you've seen in `Lab - Spark introduction (Vocareum)`, and `Homework - kNN implementation with Spark`. We need a dictionary, as an RDD, that includes the `numWords` (50 for small dataset, 20,000 for big dataset) most frequent words
in the training corpus. The result of such an RDD must be in this format:

`
[('mostcommonword', 0),
 ('nextmostcommonword', 1),
 ...]
`

Run the code cells below to create this dictionary, named `refDict`, as an RDD. This `refDict` RDD will be our reference dictionary of words. The words in `refDict` will be our reference words for which we will compute "TF-IDF" features for our training corpus and finally for the test documents.

In [ ]:
# set the number of dictionary words
# 50 for the small dataset
# 20,000 for the large dataset
numWords = 50

In [ ]:
# load up the training dataset
# "data/SmallTrainingDataOneLinePerDoc.txt" for small dataset
# "s3://comp643bucket/homework/spark_logreg/TrainingDataOneLinePerDoc.txt" for entire large dataset
corpus = sc.textFile ("SmallTrainingDataOneLinePerDoc.txt")

# each entry in validLines will be a line from the text file
corpus_validLines = corpus.filter(lambda x : 'id=' in x)

# now we transform it into a bunch of (docID, text) pairs
keyAndText = corpus_validLines.map(lambda x : (x[x.index('id="') + 4 : x.index('" url=')], x[x.index('">') + 2:x.index(' </doc>')]))

# now we split the text in each (docID, text) pair into a list of words
# after this, we have a dataset with (docID, ["word1", "word2", "word3", ...])
# we have a bit of fancy regular expression stuff here to make sure that we do not
# die on some of the documents
regex = re.compile('[^a-zA-Z]')
keyAndListOfWords = keyAndText.map(lambda x : (str(x[0]), regex.sub(' ', x[1]).lower().split()))

# now get the top numWords words... first change (docID, ["word1", "word2", "word3", ...])
# to ("word1", 1) ("word2", 1)...
allWords = keyAndListOfWords.flatMap(lambda x: ((j, 1) for j in x[1]))

# now, count all of the words, giving us ("word1", 1433), ("word2", 3423423), etc.
allCounts = allWords.reduceByKey (lambda a, b: a + b)

# and get the top numWords frequent words in a local array
topWords = allCounts.top (numWords, lambda x : x[1])

# and we'll create an RDD that has a bunch of (word, rank) pairs
# start by creating an RDD that has the number 0 up to numWords (50 for small dataset, 20K for large dataset)
# numWords is the number of words that will be in our dictionary
twentyK = sc.parallelize(range(numWords))

# now, we transform (0), (1), (2), ... to ("mostcommonword", 0) ("nextmostcommon", 1), ...
# the number will be the spot in the dictionary used to tell us where the word is located
refDict = twentyK.map(lambda x:(topWords[x][0],x))

In [ ]:
refDict.take(10)

## Provided code - TF-IDF feature extraction from the training set

Run the cell below to define the function, `compute_tfidf_training()`, which gets a training set as an RDD and a reference dictionary of words as an RDD, and computes TF-IDF feature vector of each document in the training set. This function returns `tf` as the term frequency, `idf` as the inverse document frequency, and finally `tfidf` ad the TF-IDF feature vector.   

The outputs `tf` and `tfidf` are key-value pair RDDs, while `idf` is a single 1D numpy array.

In [ ]:
def compute_tfidf_training (dataset, dictionary=refDict):

    # each entry in validLines will be a line from the text file
    validLines = dataset.filter(lambda x : 'id=' in x)

    # now we transform it into a bunch of (docID, text) pairs
    keyAndText = validLines.map(lambda x : (x[x.index('id="') + 4 : x.index('" url=')], x[x.index('">') + 2:x.index(' </doc>')]))

    # now we split the text in each (docID, text) pair into a list of words
    # after this, we have a dataset with (docID, ["word1", "word2", "word3", ...])
    # we have a bit of fancy regular expression stuff here to make sure that we do not
    # die on some of the documents
    regex = re.compile('[^a-zA-Z]')
    keyAndListOfWords = keyAndText.map(lambda x : (str(x[0]), regex.sub(' ', x[1]).lower().split()))

    # compute tf-idf

    word_docid_pair = keyAndListOfWords.flatMap(lambda x:((j,x[0]) for j in x[1]))
    joined_rdd = word_docid_pair.join(dictionary)
    docid_wordpos_pair = joined_rdd.map(lambda x: x[1]) # or: joined_rdd.values()
    grouped_docs = docid_wordpos_pair.groupByKey()

    def create_array (lst):
        arr = np.zeros(numWords)
        for word_pos in lst:
            arr[word_pos] += 1
        return arr

    bag_of_words = grouped_docs.map(lambda x: (x[0], create_array(x[1])))

    tf = bag_of_words.mapValues(lambda x: x/sum(x))

    numerator = validLines.count()
    binary_vector = bag_of_words.mapValues(lambda x: np.clip(x, 0, 1))
    denominator = binary_vector.values().reduce(lambda a, b: a + b)
    idf = np.log(numerator/denominator)

    tfidf = tf.mapValues(lambda x: x*idf)

    return tf, idf, tfidf

Call `compute_tfidf_training()` function on the `corpus`, which is our training set RDD, and save the result TF-IDF RDD as `training_tfidf`.  

In [ ]:
training_tfidf = compute_tfidf_training(corpus)[2]

In [ ]:
training_tfidf.take(3)

## Task 1 - Standardization (15 pts)

To get good accuracy, you will need to center and normalize the features. That is, transform the features so that the mean of each feature is zero, and the standard deviation is 1.

The general method of calculation is to determine the mean and standard deviation of each feature. Then, subtract the mean from each feature, and divide the result by the standard deviation.

$$\frac{x-\hat{x}}{\sigma}$$

Among TF-IDF features, you might find features with zero values for all data points. For example, the word `and`, has TF-IDF value of zero for all the documents. That's because it has zero value of IDF. That means, it appears in all documents in the corpus. For the features with a constant value for all the data points, standardization leads to 0/0 resulting in `nan`. Therefore, we ask you to check if standard deviation for any feature equals 0. If it equals 0, change standard deviation from 0 to 1. Then, standardization leads to 0/1 resulting in 0.

After standardization, to avoid overflow problems, scale down all the features by multiplying with 0.01.

For this task, start from `training_tfidf` RDD, and standardize feature vectors, following the steps below:
* Compute the mean vector of each feature (hint: you can use `.mean()` RDD action)
* Compute the standard deviation vector of each feature (hint: you can use `.stdev()` RDD action)
* If standard deviation for any feature equals 0, change 0 to 1 (to avoid division by zero error)
* Subtract the mean vector from the feature vector of each document, divide the result by the standard deviation vector, and finally multiply that with 0.01

Print out your standardized TF-IDF vector for these documents: `"AU35"` and "`7307261`". You can use `filter` RDD operation.

In [ ]:
# Start your code here ...

In [ ]:
#Compute the mean
mean_vector = training_tfidf.map(lambda x: x[1]).mean()

# Compute the standard deviation
std_vector = training_tfidf.map(lambda x: x[1]).stdev()

# Replace any standard deviation
std_vector = np.where(std_vector == 0, 1, std_vector)

# Standardize the TF-IDF vectors
standardized_tfidf = training_tfidf.mapValues(lambda x: 0.01 * ((x - mean_vector) / std_vector))

# Print the standardized TF-IDF vectors for the document AUS35"
standardized_tfidf_vector_AUS35 = standardized_tfidf.filter(lambda x: x[0] == "AU35").collect()

# Print the standardized TF-IDF vectors for the document 7307261"
standardized_tfidf_vector = standardized_tfidf.filter(lambda x: x[0] == "7307261").collect()

Print out your standardized TF-IDF vector for document `"AU35"`. You can use `filter` RDD operation.

In [ ]:
standardized_tfidf_vector_AUS35

Expected Output:

```
array([ 1.11398563e-02,  1.49570761e-02,  0.00000000e+00, -1.25991433e-02,
        1.30716062e-02, -5.17412848e-03, -8.57049570e-03, -3.62229504e-03,
        3.76559006e-03, -1.21248448e-02,  1.16486009e-03, -4.51372710e-03,
        1.91283169e-02, -9.88497388e-04,  5.16308707e-03, -6.95723678e-03,
       -5.38008764e-03, -5.43838687e-03,  5.13720970e-03, -3.70783013e-03,
        2.25466488e-02, -6.18321389e-03,  1.64058081e-02,  5.70800624e-03,
       -2.63045279e-05, -7.63143106e-03, -2.27091136e-03, -1.16710680e-04,
        1.46543592e-02, -7.85021721e-03, -8.07216165e-03, -2.61716646e-03,
       -8.47872767e-03, -3.12129299e-03, -7.71417545e-03, -5.19102945e-03,
       -2.18989926e-03, -3.80802644e-03, -9.29632877e-03, -2.66298169e-03,
       -5.40117097e-03, -8.32301255e-03, -1.24010908e-03, -3.43420202e-03,
       -5.67929991e-03, -2.24930397e-03, -9.69470215e-03, -4.55175596e-03,
        1.25484648e-02,  3.08294348e-03])
  ```

Print out your standardized TF-IDF vector for document `"7307261"`. You can use `filter` RDD operation.

In [ ]:
standardized_tfidf_vector

Expected Output:

```
array([-1.09730689e-02,  4.37845051e-03,  0.00000000e+00,  9.15695183e-03,
       -8.46654814e-03,  7.50575106e-03,  1.32081939e-02, -4.14208624e-04,
       -3.08334801e-03, -4.27615990e-03, -7.43595091e-03, -8.62312418e-03,
       -1.01472399e-02,  2.91775366e-04, -1.26379925e-03,  2.11764705e-02,
       -1.31972695e-02,  5.60245059e-03,  3.37798522e-02, -1.01503505e-02,
        6.08381323e-03, -3.84790843e-03, -6.93017782e-03, -5.58822283e-03,
       -8.28673212e-04,  3.40672466e-03, -2.50080150e-03, -7.00704334e-03,
       -1.32743947e-03,  6.70633711e-03, -5.29263577e-03, -9.28262989e-03,
        2.38277637e-03, -7.23807039e-04,  3.86659258e-03, -8.23273872e-03,
       -4.90980211e-03, -3.80802644e-03, -1.06153026e-02, -3.53521659e-03,
       -3.23836166e-05,  1.02566857e-02, -1.04810679e-02, -3.43420202e-03,
       -9.90113478e-03, -1.10533927e-02,  1.54265525e-02,  8.28684194e-03,
       -3.05985623e-03, -7.08724032e-03])
```

## Task 2 - Add intercept (10 pts)

Without adding intercept to your model (adding an extra feature with a constant value), you will probably see a poor prediction.

You can greatly improve your model by adding an intercept to your model. You can do that by adding another feature, called intercept, to your standardized training set with a constant value for all the data points. The learning process will then choose a regression coefficient for the intercept that tends to balance the “yes” and “no” nicely at a cutoff of zero.

Generally, the constant value to add is 1. But, since we scaled down the standardized features by multiplying with 0.01, for this task, **add a new feature at the end of the standardized feature vector of all documents with the constant value 0.01.** Apply this change on the standardized RDD you got from task 1.    

Print out your standardized TF-IDF vector with an intercept feature at the end for these documents: `"AU35"` and "`7307261`". You can use `filter` RDD operation. The expected result should be the same as task 1, plus having 0.01 at the end of arrays.  

In [ ]:
# Start your code here ...

In [ ]:
standardized_tfidf_with_intercept = standardized_tfidf.mapValues(lambda x: np.append(x, 0.01))

# standardized tf-idf vector with intercept
au35_vector_with_intercept = standardized_tfidf_with_intercept.filter(lambda x: x[0] == "AU35").collect()

doc_730261_vector_with_intercept = standardized_tfidf_with_intercept.filter(lambda x: x[0] == "7307261").collect()

In [ ]:
au35_vector_with_intercept

In [ ]:
doc_730261_vector_with_intercept

## Task 3 - Learning logistic regression (40 pts)

For this task, you will be implementing the gradient descent algorithm to learn a logistic regression model that can decide whether a document is describing an Australian court case or not. The algorithm needs to include Bold Driver, and uses loss function for stopping condition (i.e., stop when the change in the loss function across iterations is very small). Also, terminate the iterations if exceeds `numIters` number of iterations.   

![image.png](attachment:image.png)

As shown in the lecture, here is the loss function you need to define for your logistic regression as well as the formula for the gradient of this loss function:

![image.png](attachment:image.png)

As you can see in the equations, you need to first extract the true labels from the training set, as your $y$ data. The true labels are 1 if the document identifiers start with "AU", and are 0 if not.

$X$ is the feature vector found in the standardized training RDD with the intercept obtained from task 2.  

Some hints for computing the gradient:
* Don't use any loop
* Try to use Numpy vectorized operations as much as you can (your feature vectors are Numpy arrays)
* Use `np.exp()`
* $X.\theta$ is the dot product between two vectors of the same length. So, you can use `np.dot()`
* Think about creating this vector of gradient in one shot (don't compute the partial derivative of each parameter separately)
* You can first focus on the computation per document (inside the summation), create a Numpy array of length m per document that saves the computation inside the summation. Then, finally, summing up the arrays for all documents (e.g., with `.reduce()`) should give you the gradient.  

Finally print out

* the number of iterations actually run
* Numpy array of optimum parameters obtained from Gradient Descent
* List of (10 words, for the small dataset) / (50 words, for the big dataset) with the largest regression coefficients. These words are most strongly related with an Australian court case.

Expected optimum parameters from the small dataset:
```
[ 6.40615332e+01  7.22490429e+00  6.33648235e-01  7.13156239e+00
  3.45454873e+01 -4.66997403e+01  1.63871056e+01 -6.34003867e+00
  3.30515744e+00  2.51944859e+01  1.36064328e+01 -6.71307815e+00
  7.61643225e+01 -2.61045829e+01  2.52832197e+01  3.48471890e+00
  4.04782060e+01 -5.61829679e+01  3.29725182e+01  7.16098074e+00
  1.99955194e+01 -1.29772947e+00  4.07323151e+01  4.74213024e+01
 -5.71650491e+01 -2.25057134e+01 -1.73571310e+01  4.25329355e+01
  6.21699333e+01 -4.85541326e+01 -2.54998495e+01  1.92463742e+01
  7.88910131e+00 -6.20952879e+01 -2.23050090e+01 -8.30719112e+00
  6.95161652e+00  3.38619393e+01 -4.92814850e+01 -2.62634295e+01
 -3.13801631e+00 -2.59255120e+01  1.53432914e+01  2.69204431e+00
 -3.12659791e+01  2.62339757e+01 -3.04684079e+01 -3.30809769e+01
  2.81477095e+01 -1.35221153e+01 -6.93142416e+02]
```

Run the provided code below to initialize parameters for your gradient descent algorithm.

In [ ]:
# provided code
np.random.seed(10)

# initial parameters (+1 for intercept)
theta_init = np.random.random_sample(numWords+1)

lRate = 0.3
eps = 0.1
numIters = 60

In [ ]:
# Start your code here ...
#standardized_tfidf = training_tfidf.mapValues(lambda x: 0.01 * ((x - mean_vector) / std_vector))
#standardized_tfidf_with_intercept = standardized_tfidf.mapValues(lambda x: np.append(x, 0.01))
data_with_labels = standardized_tfidf_with_intercept.map(lambda x: (1 if x[0].startswith("AU") else 0, x[1])).collect()
labels = np.array([label for label, _ in data_with_labels])
features = np.array([features for _, features in data_with_labels])

print("Labels:", labels[:10])
print("Features:", features[:10])

In [ ]:
# logistic regression functions
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def compute_loss(X, y, theta):
    m = len(y)
    h = sigmoid(X.dot(theta))
    epsilon = 1e-5
    loss = -(1/m) * (np.sum(y * np.log( h + epsilon) + (1-y) * np.log(1 -h + epsilon)))
    return loss

def compute_gradient(X, y, theta):
    m = len(y)
    h = sigmoid(X.dot(theta))
    gradient = (1/m) * X.T.dot(h - y)
    return gradient



# Gradient Descent with Bold Driver
theta = theta_init
loss_history = []
iterations = 0

for i in range(numIters):
    iterations += 1
    gradient = compute_gradient(features, labels, theta)
    new_theta = theta - lRate * gradient
    loss_old = compute_loss(features, labels, theta)
    loss_new = compute_loss(features, labels, new_theta)

    if loss_new < loss_old:
        lRate *= 1.2
        theta = new_theta
    else:
        lRate *= 0.5
        thetha = new_thetha

    loss_history.append(loss_new)

    if np.abs(loss_old - loss_new) < eps:
        break
print("Number of iterations:", iterations)

# Numpy array of optimum parameters obtained from Gradient Descent
print("Optimum parameters:", theta)

# List of 10 words with the largest regression coefficients
top_indices = np.argsort(theta[:-1])[-10:]  # Excluding the intercept
top_words = [word for word, index in refDict.collect() if index in top_indices]

print("Top 10 words with the largest regression coefficients:", top_words)


## Task 4 - prediction (35 pts)

Now that you have trained your model and found the optimum parameters, it is time to evaluate it.

For the last part of this homework, your task is to apply your model on the provided test set, and get the predictions. Use the cutoff $ X.\theta = 0$

If $ X.\theta > 0$ , classify as 1 (Australian court case)  
If $ X.\theta <= 0$ , classify as 0 (Not an Australian court case)  

To extract and prepare the feature vectors of the test set,
* use the provided function `compute_tfidf_test` below.
* then, similar to task1, standardize the feature vectors of the test set. **Just note to use the same mean and std from the training set.**
* then, similar to task 2, add an extra feature at the end of the feature vector with the constant value 0.01 for all the test documents.   

To evaluate the performance of your model, compute the F1 score based on the predicted classes and the actual classes found in the test document identifiers.   

![image.png](attachment:image.png)


$precision = \frac{TP}{TP+FP}$

$recall = \frac{TP}{TP+FN}$

$F1-score = \frac{2*precision*recall}{precision+recall}$

Finally print out the evaluation results: precision, recall, and F1-score.

### Provided code - TF-IDF feature extraction from test set

Run the code cells below to define a function that computes TF-IDF feature vector of every document in the test set, and call this function on the provided test set.

In [ ]:
def compute_tfidf_test (dataset, dictionary=refDict):

    # each entry in validLines will be a line from the text file
    validLines = dataset.filter(lambda x : 'id=' in x)

    # now we transform it into a bunch of (docID, text) pairs
    keyAndText = validLines.map(lambda x : (x[x.index('id="') + 4 : x.index('" url=')], x[x.index('">') + 2:x.index(' </doc>')]))

    # now we split the text in each (docID, text) pair into a list of words
    # after this, we have a dataset with (docID, ["word1", "word2", "word3", ...])
    # we have a bit of fancy regular expression stuff here to make sure that we do not
    # die on some of the documents
    regex = re.compile('[^a-zA-Z]')
    keyAndListOfWords = keyAndText.map(lambda x : (str(x[0]), regex.sub(' ', x[1]).lower().split()))

    # compute tf-idf

    word_docid_pair = keyAndListOfWords.flatMap(lambda x:((j,x[0]) for j in x[1]))
    joined_rdd = word_docid_pair.join(dictionary)
    docid_wordpos_pair = joined_rdd.map(lambda x: x[1]) # or: joined_rdd.values()
    grouped_docs = docid_wordpos_pair.groupByKey()

    def create_array (lst):
        arr = np.zeros(numWords)
        for word_pos in lst:
            arr[word_pos] += 1
        return arr

    bag_of_words = grouped_docs.map(lambda x: (x[0], create_array(x[1])))

    tf = bag_of_words.mapValues(lambda x: x/sum(x))

    idf = compute_tfidf_training(corpus)[1]

    tfidf = tf.mapValues(lambda x: x*idf)

    return tf, idf, tfidf

In [ ]:
# load up the test dataset
# "data/SmallTestingDataOneLinePerDoc.txt" for small dataset
# "s3://comp643bucket/homework/spark_logreg/TestingDataOneLinePerDoc.txt" for entire large dataset
test = sc.textFile ("data/SmallTestingDataOneLinePerDoc.txt")
test_tfidf = compute_tfidf_test(test)[2]

In [ ]:
# Strat your code here ...
# Standardize the TF-IDF vectors of the test set
standardized_test_tfidf = test_tfidf.mapValues(lambda x: (x - mean_vector) / std_vector)

# Add intercept term to the test set features
standardized_test_tfidf_with_intercept = standardized_test_tfidf.mapValues(lambda x: np.append(0.01 * x, 0.01))

In [ ]:
test_data_with_labels = standardized_test_tfidf_with_intercept.map(lambda x: (1 if x[0].startswith("AU") else 0, x[1])).collect()
test_labels = np.array([label for label, _ in test_data_with_labels])
test_features = np.array([features for _, features in test_data_with_labels])
# Predict the classes
predictions = (test_features.dot(theta) > 0).astype(int)

# Calculate True Positives (TP), False Positives (FP), True Negatives (TN), False Negatives (FN)
TP = np.sum((predictions == 1) & (test_labels == 1))
FP = np.sum((predictions == 1) & (test_labels == 0))
TN = np.sum((predictions == 0) & (test_labels == 0))
FN = np.sum((predictions == 0) & (test_labels == 1))
# Calculate precision, recall, and F1-score
precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1_score = 2 * precision * recall / (precision + recall)

# Print evaluation results
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1_score)


## Task 5 - run on the entire dataset in EMR cluster (5 pts)

Finally, you need to run your Spark code for tasks 1 through 4, on the entire dataset stored in S3, in an AWS EMR cluster.

Follow the instructions on `Lab - Spark Intro (AWS)` to create and connect to an EMR cluster in AWS and run Spark programs in there.

You can gather your code for each task in a Python `.py` file and submit them as jobs in the batch mode and get the final result back. To troubleshoot, you can run your code line by line, in an interactive mode to debug your program.     

**Don't forget to set `numWords` to 20,000 and replace datasets directory to S3 locations.**

The entire dataset exists in this S3 URI:

* The entire training set (~ 1.8 GB of text): `s3://comp643bucket/homework/spark_logreg/TrainingDataOneLinePerDoc.txt`

* The entire test set (~ 200 MB of text): `s3://comp643bucket/homework/spark_logreg/TestingDataOneLinePerDoc.txt`

**Choose 6 instances of `m5.xlarge` machines in AWS Academy EMR. For your reference, our solution run time on this cluster configuration is as follows: task 1 ~ 4 min, task 2 ~ 2 min, task 3 ~ 42 min, task 4 ~ 6 min.**

If you find that your training takes more than a few hours to run on the big dataset, it likely means that you are doing something that is inherently slow that you can speed up by looking at your code carefully. In addition, here are some hints that can help your code run faster:

1. You might want to explicitly partition the data to fully make use of your powerful machines, by setting `minPartitions` parameter in `sc.textFile()`. This parameter sets minimum number of partitions for the resulting RDD. As an example, you can set this to 16.

2. Think about caching RDD(s), through `.cache()` method. Although it is simply a function call of `.cache()`, you need to think about which RDD(s) to cache. It obviously does not make sense to cache everything. You can cache RDDs that are frequently called.

Repeat tasks 1 through 4 on the entire dataset in your EMR cluster, and print your results in the markdown cells below (keep the results from the small subset above).

Task 1 - Print out your standardized TF-IDF vector for these documents: `"AU35"` and `"7307261"`. It's fine if the print out you get only shows the first and last 3 entries. Just copy the result here.

...

Task 2 - Print out your standardized TF-IDF vector with an intercept feature at the end for these documents: "AU35" and "7307261". It's fine if the print out you get only shows the first and last 3 entries. Just copy the result here.

...

Task 3 - Print out, the number of iterations actually run, Numpy array of optimum parameters obtained from Gradient Descent, List of 50 words with the largest regression coefficients. These words are most strongly related with an Australian court case.

...

Task 4 - Print out the evaluation results: precision, recall, and F1-score

...